# Real DICOM CT → TotalSegmentator → OHIF bundle

This notebook turns **one de-identified DICOM CT series** into a standards-valid DICOM SEG that Clinique Amina/OHIF can display.

Before running it:

1. Put a de-identified CT DICOM folder or ZIP in Google Drive.
2. In the configuration cell, set `DICOM_INPUT` to that path.
3. Set `CONFIRM_DEIDENTIFIED = True` only after you have confirmed the scan contains no patient identity.

> This workflow segments anatomy for visual quality control. It is not a diagnosis or a medical device.


In [ ]:
from pathlib import Path
import hashlib
import subprocess
import sys

REPO_URL = "https://github.com/Hamza09Hamza/doctor_assistant.git"
REPO_REF = "classifier-evaluation-colab"
REPO_DIR = Path("/content/doctor_assistant")

if REPO_DIR.exists():
    status = subprocess.run(
        ["git", "status", "--porcelain"], cwd=REPO_DIR, capture_output=True, text=True, check=True
    )
    if status.stdout.strip():
        raise RuntimeError(f"{REPO_DIR} has local changes. Remove that Colab checkout and run again.")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "switch", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{REPO_REF}"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)

local_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
remote_commit = subprocess.check_output(["git", "rev-parse", f"origin/{REPO_REF}"], cwd=REPO_DIR, text=True).strip()
if local_commit != remote_commit:
    raise RuntimeError(f"Repository mismatch: local {local_commit} != remote {remote_commit}")
script_path = REPO_DIR / "scripts/run_totalsegmentator_dicom_seg.py"
print("Repository commit:", local_commit)
print("DICOM SEG script SHA-256:", hashlib.sha256(script_path.read_bytes()).hexdigest())


In [ ]:
# TotalSegmentator 2.17 introduced direct DICOM SEG output.
%pip install -q "numpy>=1.26,<2.1" "TotalSegmentator==2.17.0" "pydicom>=3.0,<4" "highdicom==0.27.0"

import importlib.metadata as metadata
print("TotalSegmentator:", metadata.version("TotalSegmentator"))
print("highdicom:", metadata.version("highdicom"))
print("pydicom:", metadata.version("pydicom"))


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → L4 GPU, then run again.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# CHANGE THIS to your de-identified DICOM folder or ZIP in Drive.
DICOM_INPUT = Path("/content/drive/MyDrive/doctor_assistant/dicom_input/ct_series.zip")
CONFIRM_DEIDENTIFIED = False  # Change to True only after checking the input.
SERIES_INSTANCE_UID = ""   # Leave empty unless the script reports multiple CT series.
USE_FAST_MODEL = False        # Full 1.5 mm model is recommended on an L4.
FORCE_RERUN = False

OUTPUT_DIR = Path("/content/drive/MyDrive/doctor_assistant/totalsegmentator_dicom_seg/results")
WORK_DIR = Path("/content/totalsegmentator_dicom_work")
# Reuse the model weights already downloaded by the successful visual-proof notebook.
WEIGHTS_DIR = Path("/content/drive/MyDrive/doctor_assistant/totalsegmentator_demo/weights")
for directory in (OUTPUT_DIR, WORK_DIR, WEIGHTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if not DICOM_INPUT.exists():
    raise FileNotFoundError(f"Set DICOM_INPUT to your real Drive path: {DICOM_INPUT}")
if not CONFIRM_DEIDENTIFIED:
    raise RuntimeError("Confirm the scan is de-identified, then set CONFIRM_DEIDENTIFIED = True.")
print("Input ready. Patient-identifying tag values will not be printed.")


In [ ]:
import os

command = [
    sys.executable, "-u", "scripts/run_totalsegmentator_dicom_seg.py",
    "--dicom-input", str(DICOM_INPUT),
    "--output-dir", str(OUTPUT_DIR),
    "--work-dir", str(WORK_DIR),
    "--confirm-deidentified",
]
if SERIES_INSTANCE_UID:
    command += ["--series-instance-uid", SERIES_INSTANCE_UID]
if USE_FAST_MODEL:
    command.append("--fast")
if FORCE_RERUN:
    command.append("--force")

env = {**os.environ, "PYTHONUNBUFFERED": "1", "TOTALSEG_WEIGHTS_PATH": str(WEIGHTS_DIR)}
print("Command:", " ".join(command))
process = subprocess.Popen(
    command, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()
if return_code:
    raise RuntimeError(f"DICOM SEG process exited with code {return_code}")


In [ ]:
import json

latest = json.loads((OUTPUT_DIR / "latest_dicom_seg_run.json").read_text())
manifest = json.loads(Path(latest["manifest"]).read_text())
assert manifest["validation"]["segment_count"] > 0
assert Path(latest["dicom_seg"]).exists()
assert Path(latest["viewer_bundle"]).exists()
print("SUCCESS — DICOM SEG references and pixel data passed validation.")
print("Non-empty anatomical segments:", manifest["validation"]["segment_count"])
print("OHIF viewer bundle:", latest["viewer_bundle"])
print("Download that ZIP to the computer where you will run Orthanc/OHIF.")


## Then open it in Clinique Amina

On your local computer, start the viewer stack and publish the ZIP using the commands printed in `docs/TOTALSEGMENTATOR_OHIF.md`. The publisher prints one exact Clinique Amina URL for the study.
